# Training of the student

What we do:
1) Load the training dataset
2) Run training using the huggingface client
3) Save the model


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_training_client import HuggingFaceTrainingClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from core.distillation.sampling import stratified_sampling_with_features
from sklearn.cluster import DBSCAN
from transformers import AutoTokenizer
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import os
import json
import numpy as np
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
dataset1_path = Path("data/outputs/selected-data-low-reasoning-gpt5.csv")
dataset2_path = Path("data/outputs/non-selected-data-low-reasoning-gpt5.csv")

df1 = pd.read_csv(dataset1_path)
df2 = pd.read_csv(dataset2_path)
df = pd.concat([df1, df2])
df.dropna(axis=0, how='all', inplace=True)

In [5]:
df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-233-p2-uc0,AIMED_AT:\n type: Wall\n distance: 330.86\n ...,Walk up to that switch and use it,Approach the interactable wall and trigger its...,0.85,0.35,0.90,MOVE 0.0 330.86\nINTERACT,4.414740,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,state-6603-p1-uc0,AIMED_AT:\n type: Wall\n distance: 281.96\n ...,Go press that switch ahead,Approach the interactable wall in front and ac...,0.90,0.40,0.85,SPRINT 0.0 281.96\nINTERACT,3.383618,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,state-7768-p3-uc1,AIMED_AT:\n type: Wall\n distance: 736.10\n ...,Turn left and kill the enemy,Rotate toward the visible monster and eliminat...,0.80,0.40,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 3,2.579986,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,state-7825-p3-uc1,AIMED_AT:\n type: Wall\n distance: 675.93\n ...,Turn left and blast the closest guy,Face and kill the nearest zombieman threatenin...,0.85,0.50,0.90,ROTATE_TO_TARGET MONSTER_0\nFIRE 0.7,4.923402,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,state-8097-p0-uc0,AIMED_AT:\n type: Wall\n distance: 783.61\n ...,Drop the closest zombie with the shotgun,Quickly kill the nearest low-health zombieman ...,0.85,0.40,0.90,ROTATE_TO_TARGET MONSTER_1\nFIRE_SHOTS 1,4.482084,NaN,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,state-58188-p0-uc1,AIMED_AT:\n type: Monster\n distance: 301.90...,Back up while firing at the skull,Kite the charging lost soul to stay safe while...,0.80,0.40,0.90,ROTATE_TO_TARGET MONSTER_0\nASYNC FIRE 2.0\nMO...,0.000000,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2818,state-58188-p0-uc2,AIMED_AT:\n type: Monster\n distance: 301.90...,Conserve ammo and just dodge it,Avoid damage from the lost soul without spendi...,0.75,0.50,0.85,SPRINT 100.0 0.0,0.000000,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2819,state-58188-p1-uc0,AIMED_AT:\n type: Monster\n distance: 301.90...,Put that lost soul in my crosshair,Align the aim precisely with the approaching l...,0.85,0.35,0.85,ROTATE_TO_TARGET MONSTER_0,0.000000,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2820,state-58188-p1-uc1,AIMED_AT:\n type: Monster\n distance: 301.90...,Shoot the flying skull now,Immediately fire the pistol to damage the visi...,0.90,0.80,0.80,ROTATE_TO_TARGET MONSTER_0\nFIRE 1.0,0.000000,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [6]:
# Extract only <inputs, labels>
# For inputs, we do not want (nor need) the full prompt. Just a couple of general pieces of information.
# That is, the inputs are SMALL PROMPT + GAME_STATE + USER COMMAND
# The labels are the df.labels

dataset = [
    TrainingEntry(
        game_state=row.game_state,
        user_command=row.command,
        expected_actions=row.game_actions,
        metrics=TrainingEntryMetrics(
            explicitness=row.command_explicitness,
            atomicity=row.command_atomicity,
            contextuality=row.command_contextuality,
            cluster_id=row.cluster_id
        )
    )
    for row in df.itertuples()
]

In [7]:
training_client = HuggingFaceTrainingClient[TrainingEntry](
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device="cuda",
    working_dir=Path("data/huggingface"),
    use_qlora=False,
    use_flash_attention_2=False
)

🏋️  Initialized HuggingFaceTrainingClient for Qwen/Qwen2.5-1.5B-Instruct
   Device: cuda
   QLoRA: False
   Flash Attention 2: False


In [16]:
def format_example(example: TrainingEntry, tokenizer: AutoTokenizer) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a game command parser that converts natural language commands into DSL instructions."
        },
        {
            "role": "user",
            "content": f"Game State: {example.game_state}\nCommand: {example.user_command}"
        },
        {
            "role": "assistant",
            "content": example.expected_actions
        }
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        # Template already adds EOS - just return it
        return formatted.rstrip()

    return json.dumps(messages) + tokenizer.eos_token

In [9]:
# Sample based on the metrics we have for each example

train_data, eval_data = stratified_sampling_with_features(
    dataset,
    eval_ratio=0.2
)

📊 Stratified sampling by features:
   Total clusters: 10
   Cluster 7.0: 4 train, 1 eval
   Cluster 4.0: 3 train, 2 eval
   Cluster 5.0: 4 train, 1 eval
   Cluster 9.0: 4 train, 1 eval
   Cluster 3.0: 4 train, 1 eval
   Cluster 2.0: 3 train, 2 eval
   Cluster 1.0: 4 train, 1 eval
   Cluster 0.0: 4 train, 1 eval
   Cluster 8.0: 4 train, 1 eval
   Cluster 6.0: 4 train, 1 eval


In [11]:
training_client.load_tokenizer()


📦 Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
   ✓ Tokenizer loaded successfully!



In [17]:
sample = format_example(train_data[0], training_client.tokenizer)
print(f"EOS token: {repr(training_client.tokenizer.eos_token)}")
print(f"Last 50 chars: {repr(sample[-50:])}")
print(f"Ends with EOS? {sample.endswith(training_client.tokenizer.eos_token)}")

# Also test what the template gives you BEFORE rstrip
messages = [
    {
        "role": "system",
        "content": "You are a game command parser that converts natural language commands into DSL instructions."
    },
    {
        "role": "user",
        "content": f"Game State: {train_data[0].game_state}\nCommand: {train_data[0].user_command}"
    },
    {
        "role": "assistant",
        "content": train_data[0].expected_actions
    }
]

formatted_raw = training_client.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)
print(f"\nBefore rstrip: {repr(formatted_raw[-50:])}")
print(f"After rstrip: {repr(formatted_raw.rstrip()[-50:])}")

EOS token: '<|im_end|>'
Last 50 chars: '\nROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 2<|im_end|>'
Ends with EOS? True

Before rstrip: 'ROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 2<|im_end|>\n'
After rstrip: '\nROTATE_TO_TARGET MONSTER_0\nFIRE_SHOTS 2<|im_end|>'


In [14]:
training_client.fine_tune(
    train_dataset=train_data,
    eval_dataset=eval_data,
    format_example=format_example,
    output_dir=Path("data/huggingface/training")
)


📦 Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
   ✓ Tokenizer loaded successfully!


📦 Loading model for training: Qwen/Qwen2.5-1.5B-Instruct
   ✓ Model loaded successfully!

🔧 Fine-tuning configuration:
   Output dir: data\huggingface\training
   Epochs: 3
   Batch size: 4
   Gradient accumulation: 4
   Effective batch size: 16
   Learning rate: 0.0002
   Max sequence length: 2048
📝 Formatting 38 training examples...
📝 Formatting 12 eval examples...
🔤 Creating and tokenizing datasets...


Tokenizing training data:   0%|          | 0/38 [00:00<?, ? examples/s]

Tokenizing eval data:   0%|          | 0/12 [00:00<?, ? examples/s]

   ✓ Datasets prepared

🚀 Starting fine-tuning...



`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss



✅ Fine-tuning complete! Model saved to data\huggingface\training\final


WindowsPath('data/huggingface/training/final')